# Part II — CHS Features and Economic Reasoning

FNCE 2370 — Team 7 | Predicting Corporate Distress Risk

Validate supplied predictors, define common eligibility, screen additional characteristics on initial training coverage, and fit training-only winsorization and imputation parameters.

Run from the repository root (or its `notebooks/` subdirectory). See `README.md` for execution instructions. Source-data assumptions remain provisional pending TA confirmation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.6f}".format)

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

file_path = project_root / 'Project_5_data_chs_main_global.parquet'

processed_dir = project_root / "data" / "processed"
tables_dir = project_root / "outputs" / "tables"

panel_path = processed_dir / "part_I_panel.parquet"

assert file_path.exists(), "Raw dataset not found."
assert panel_path.exists(), "Part I panel not found."

print("Input files validated.")

figures_dir = project_root / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

## Baseline features and economic motivation

Use the eight preferred CHS (2008) predictors, as required by the assignment. The table gives the paper's definitions and expected economic relationships, not constraints on estimated coefficients. Let market-valued total assets (MTA) equal market equity plus book total liabilities.

| Predictor | CHS definition / economic rationale | Expected relation to failure |
|---|---|---|
| `nimtaavg` | Weighted history of net income / MTA; persistent losses weaken financial health | − |
| `tlmta` | Total liabilities / MTA; greater obligations increase risk | + |
| `exretavg` | Weighted history of log excess returns relative to the S&P 500; poor performance signals deterioration | − |
| `sigma` | Annualized zero-centered daily return volatility over three months; equity uncertainty | + |
| `rsize` | Log firm market equity relative to total S&P 500 market equity; financing capacity | Usually − unconditionally; conditional sign may differ |
| `cashmta` | Cash and short-term investments / MTA; liquidity buffer | − |
| `mb` | Market equity / adjusted book equity; valuation and book-equity erosion | Ambiguous (positive in CHS one-month model) |
| `price_log` | Log absolute share price capped at $15; low prices signal distress | − |

CHS uses a two-month accounting lag, quarterly-block weighted NIMTA, monthly weighted EXRET, and a three-month half-life. It adjusts book equity for extreme values and uses a zero-centered volatility calculation. We retain the supplied variables, validate feasible identities and histories below, and do not impose an arbitrary second accounting lag. Publication dates, stale accounting values, underlying daily returns and book-equity adjustments cannot be independently verified from this panel.

**Timing assumption:** the supplied month-end features are treated as observable at that month-end, conditional on the instructor's upstream construction. The schema includes no separate fiscal-period or announcement-date field. Repeated ratios cannot identify accounting staleness because the denominator can change monthly. We therefore cannot infer a valid lag or freshness screen from these values alone. The original source documentation is needed to confirm this assumption.

Supplementary Parquet inspection found no file or schema provenance metadata or identifiable accounting release/fiscal-date fields. The Global Characteristics schema is consistent with the JKP library, but the precise release and its accounting availability/staleness conventions are unconfirmed. Do not assume that published JKP defaults or the CHS two-month lag apply to this file. Await source documentation; our own training-only transformations do not correct any upstream timing error.

In [ ]:
CHS_FEATURES = [
    "nimtaavg",
    "tlmta",
    "exretavg",
    "sigma",
    "rsize",
    "cashmta",
    "mb",
    "price_log"
]

AUXILIARY_FEATURES = [
    "nimta",
    "nita",
    "tlta",
    "exret",
    "sigma_fill",
    "prc",
    "market_equity",
    "ret_adj",
    "sprtrn"
]

KEYS = ["permno", "month_end"]

print("Number of CHS predictors:", len(CHS_FEATURES))
print("Number of auxiliary variables:", len(AUXILIARY_FEATURES))

In [ ]:
panel = pd.read_parquet(panel_path)

features = pd.read_parquet(
    file_path,
    columns=KEYS + CHS_FEATURES + AUXILIARY_FEATURES
)

features["month_end"] = pd.to_datetime(
    features["month_end"]
)

assert not features.duplicated(KEYS).any()
assert not panel.duplicated(KEYS).any()

df = panel.merge(
    features,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(df) == len(panel)
assert df["_merge"].eq("both").all()

df = df.drop(columns="_merge")

df = df.sort_values(
    ["permno", "feature_month"]
).reset_index(drop=True)

del features, panel

print("Dataset shape:", df.shape)
print("Labeled observations:", df["labeled"].sum())

print(
    "Memory usage (GB):",
    round(df.memory_usage(deep=True).sum() / 1e9, 3)
)

## Development-only diagnostics

Use forecast-month cutoffs (Part III) before estimating any transformation or screening characteristics. Initial training ends in December 1995; validation covers 1996–2003. Distributional and failure-group diagnostics are based on training data, with validation coverage used descriptively. The final 2004–2024 period is not used to select features or estimate preprocessing parameters.

In [ ]:
TRAIN_END = pd.Timestamp("1995-12-31")
VALIDATION_END = pd.Timestamp("2003-12-31")
TEST_END = pd.Timestamp("2024-12-31")

train_mask = (
    df["labeled"]
    & df["forecast_month"].le(TRAIN_END)
)

validation_mask = (
    df["labeled"]
    & df["forecast_month"].gt(TRAIN_END)
    & df["forecast_month"].le(VALIDATION_END)
)

development_mask = train_mask | validation_mask

print("Initial training observations:", train_mask.sum())
print("Validation observations:", validation_mask.sum())
print("Development observations:", development_mask.sum())

In [ ]:
train_features = df.loc[
    train_mask,
    CHS_FEATURES
].replace([np.inf, -np.inf], np.nan)

feature_summary = train_features.describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99
    ]
).T

feature_summary["missing_pct"] = (
    train_features.isna().mean() * 100
)

display(feature_summary)

In [ ]:
missingness = pd.DataFrame({
    "training_missing_pct":
        df.loc[train_mask, CHS_FEATURES]
        .isna().mean() * 100,

    "validation_missing_pct":
        df.loc[validation_mask, CHS_FEATURES]
        .isna().mean() * 100,

    "training_nonfailure_missing_pct":
        df.loc[
            train_mask & df["target"].eq(0),
            CHS_FEATURES
        ].isna().mean() * 100,

    "training_failure_missing_pct":
        df.loc[
            train_mask & df["target"].eq(1),
            CHS_FEATURES
        ].isna().mean() * 100
})

display(missingness)

In [ ]:
train = df.loc[train_mask]

unusual_values = pd.Series({
    "Negative TLMTA":
        train["tlmta"].lt(0).sum(),

    "TLMTA above one":
        train["tlmta"].gt(1).sum(),

    "Negative CASHMTA":
        train["cashmta"].lt(0).sum(),

    "CASHMTA above one":
        train["cashmta"].gt(1).sum(),

    "Nonpositive MB":
        train["mb"].le(0).sum(),

    "Nonpositive SIGMA":
        train["sigma"].le(0).sum(),

    "SIGMA above five":
        train["sigma"].gt(5).sum(),

    "PRICE above log(15)":
        train["price_log"].gt(
            np.log(15) + 1e-6
        ).sum(),

    "Infinite CHS values":
        np.isinf(
            train[CHS_FEATURES]
            .to_numpy(dtype=float)
        ).sum()
})

display(unusual_values)

## Validate supplied feature construction

Test PRICE against the supplied share price and EXRET against log gross returns minus log gross S&P 500 returns. The accounting-ratio identity below checks consistency of shared numerators and denominators, not the original accounting release dates. Use raw `sigma` rather than `sigma_fill` so the imputation rule is explicitly estimated on our training sample.

In [ ]:
valid_price = (
    train["prc"].notna()
    & train["prc"].ne(0)
    & train["price_log"].notna()
)

expected_price = np.log(
    train.loc[valid_price, "prc"]
    .abs()
    .clip(upper=15)
)

price_error = (
    train.loc[valid_price, "price_log"]
    - expected_price
).abs()

price_validation = pd.Series({
    "Comparable observations": len(price_error),

    "Matching observations":
        price_error.le(1e-6).sum(),

    "Mismatched observations":
        price_error.gt(1e-6).sum(),

    "Mean absolute error": price_error.mean(),

    "Maximum absolute error": price_error.max()
})

display(price_validation)

In [ ]:
valid_returns = (
    train["ret_adj"].gt(-1)
    & train["sprtrn"].gt(-1)
    & train["exret"].notna()
)

expected_exret = (
    np.log1p(train.loc[valid_returns, "ret_adj"])
    - np.log1p(train.loc[valid_returns, "sprtrn"])
)

exret_error = (
    train.loc[valid_returns, "exret"]
    - expected_exret
).abs()

exret_validation = pd.Series({
    "Comparable observations": len(exret_error),

    "Matching observations":
        exret_error.le(1e-6).sum(),

    "Mismatched observations":
        exret_error.gt(1e-6).sum(),

    "Mean absolute error": exret_error.mean(),

    "Maximum absolute error": exret_error.max()
})

display(exret_validation)

In [ ]:
observed_sigma = train["sigma"].notna()

sigma_difference = (
    train.loc[observed_sigma, "sigma"]
    - train.loc[observed_sigma, "sigma_fill"]
).abs()

sigma_diagnostics = pd.Series({
    "Observed SIGMA":
        train["sigma"].notna().sum(),

    "Missing SIGMA":
        train["sigma"].isna().sum(),

    "Missing SIGMA_FILL":
        train["sigma_fill"].isna().sum(),

    "Missing SIGMA replaced by SIGMA_FILL":
        (
            train["sigma"].isna()
            & train["sigma_fill"].notna()
        ).sum(),

    "Observed SIGMA altered by SIGMA_FILL":
        sigma_difference.gt(1e-6).sum()
})

display(sigma_diagnostics)

imputed_sigma = train.loc[
    train["sigma"].isna()
    & train["sigma_fill"].notna(),
    ["feature_month", "sigma_fill"]
]

display(imputed_sigma["sigma_fill"].describe())

In [ ]:
monthly_sigma_mean = train.groupby(
    "feature_month"
)["sigma"].mean()

expected_fill = imputed_sigma[
    "feature_month"
].map(monthly_sigma_mean)

fill_error = (
    imputed_sigma["sigma_fill"] - expected_fill
).abs()

display(fill_error.describe())

print(
    "Share matching monthly cross-sectional mean:",
    f"{fill_error.le(1e-6).mean():.4%}"
)

In [ ]:
accounting_diagnostics = pd.Series({
    "Missing current NIMTA":
        train["nimta"].isna().sum(),

    "Missing NIMTAAVG":
        train["nimtaavg"].isna().sum(),

    "Missing NIMTA with available NIMTAAVG":
        (
            train["nimta"].isna()
            & train["nimtaavg"].notna()
        ).sum(),

    "Missing all three accounting ratios":
        train[
            ["tlmta", "cashmta", "mb"]
        ].isna().all(axis=1).sum(),

    "Complete CHS observations":
        train_features.notna().all(axis=1).sum()
})

display(accounting_diagnostics)

In [ ]:
group_means = (
    train.groupby("target")[CHS_FEATURES]
    .mean()
    .T
)

group_means.columns = [
    "Non-failure",
    "Failure"
]

group_means["Difference"] = (
    group_means["Failure"]
    - group_means["Non-failure"]
)

display(group_means)

In [ ]:
group_medians = (
    train.groupby("target")[CHS_FEATURES]
    .median()
    .T
)

group_medians.columns = [
    "Non-failure",
    "Failure"
]

group_medians["Difference"] = (
    group_medians["Failure"]
    - group_medians["Non-failure"]
)

display(group_medians)

In [ ]:
valid_accounting = (
    train["nimta"].notna()
    & train["nita"].notna()
    & train["tlta"].gt(0)
    & train["tlmta"].notna()
)

expected_nimta = (
    train.loc[valid_accounting, "nita"]
    * train.loc[valid_accounting, "tlmta"]
    / train.loc[valid_accounting, "tlta"]
)

nimta_error = (
    train.loc[valid_accounting, "nimta"]
    - expected_nimta
).abs()

accounting_validation = pd.Series({
    "Comparable observations": len(nimta_error),
    "Matches within 1e-6": nimta_error.le(1e-6).sum(),
    "Matches within 1e-4": nimta_error.le(1e-4).sum(),
    "Mean absolute error": nimta_error.mean(),
    "Median absolute error": nimta_error.median(),
    "99th percentile error": nimta_error.quantile(0.99),
    "Maximum absolute error": nimta_error.max()
})

display(accounting_validation)

## Rolling and weighted features

Use the unrestricted historical security panel before eligibility screens so removed observations do not create artificial return gaps. Validate complete consecutive histories using current and past feature-month observations only. The monthly weights are proportional to $\phi^j$ for $j=0,\ldots,11$, with $\phi=2^{-1/3}$ (a three-month half-life). Compare the original CHS quarterly-block NIMTA convention with the supplied series; incomplete-history treatment is not established by these tests.

In [ ]:
history = pd.read_parquet(
    file_path,
    columns=[
        "permno",
        "month_end",
        "exret",
        "exretavg",
        "nimta",
        "nimtaavg"
    ]
)

history["month_end"] = pd.to_datetime(
    history["month_end"]
)

# Use only information preceding the training cutoff
history = history.loc[
    history["month_end"] < TRAIN_END
].copy()

history = history.sort_values(
    ["permno", "month_end"]
).reset_index(drop=True)

phi = 2 ** (-1 / 3)

weights = phi ** np.arange(12)
weights = weights / weights.sum()

weight_table = pd.DataFrame({
    "lag_month": np.arange(12),
    "weight": weights
})

display(weight_table)
print("Sum of weights:", weights.sum())

In [ ]:
history["month_index"] = (
    history["month_end"].dt.year * 12
    + history["month_end"].dt.month
)

groups = history.groupby("permno", sort=False)

weighted_exret = np.zeros(len(history))
valid_history = np.ones(len(history), dtype=bool)

for lag, weight in enumerate(weights):

    lagged_return = groups["exret"].shift(lag)
    lagged_month = groups["month_index"].shift(lag)

    consecutive = (
        history["month_index"] - lagged_month
    ).eq(lag)

    valid_history &= (
        lagged_return.notna() & consecutive
    ).to_numpy()

    weighted_exret += (
        weight * lagged_return.fillna(0).to_numpy()
    )

history["exretavg_reconstructed"] = np.where(
    valid_history,
    weighted_exret,
    np.nan
)

comparable = (
    history["exretavg_reconstructed"].notna()
    & history["exretavg"].notna()
)

exretavg_error = (
    history.loc[comparable, "exretavg"]
    - history.loc[comparable, "exretavg_reconstructed"]
).abs()

exretavg_validation = pd.Series({
    "Comparable observations": comparable.sum(),
    "Matches within 1e-6": exretavg_error.le(1e-6).sum(),
    "Matches within 1e-4": exretavg_error.le(1e-4).sum(),
    "Mean absolute error": exretavg_error.mean(),
    "Median absolute error": exretavg_error.median(),
    "99th percentile error": exretavg_error.quantile(0.99),
    "Maximum absolute error": exretavg_error.max()
})

display(exretavg_validation)

In [ ]:
quarter_weights = phi ** (3 * np.arange(4))
quarter_weights /= quarter_weights.sum()

display(pd.DataFrame({
    "quarter_lag": np.arange(4),
    "weight": quarter_weights
}))

In [ ]:
monthly_weights = np.repeat(
    quarter_weights / 3,
    3
)

weighted_nimta = np.zeros(len(history))
valid_nimta_history = np.ones(
    len(history),
    dtype=bool
)

for lag, weight in enumerate(monthly_weights):

    lagged_nimta = groups["nimta"].shift(lag)
    lagged_month = groups["month_index"].shift(lag)

    consecutive = (
        history["month_index"] - lagged_month
    ).eq(lag)

    valid_nimta_history &= (
        lagged_nimta.notna() & consecutive
    ).to_numpy()

    weighted_nimta += (
        weight * lagged_nimta.fillna(0).to_numpy()
    )

history["nimtaavg_reconstructed"] = np.where(
    valid_nimta_history,
    weighted_nimta,
    np.nan
)

comparable = (
    history["nimtaavg_reconstructed"].notna()
    & history["nimtaavg"].notna()
)

nimtaavg_error = (
    history.loc[comparable, "nimtaavg"]
    - history.loc[comparable, "nimtaavg_reconstructed"]
).abs()

nimtaavg_validation = pd.Series({
    "Comparable observations": comparable.sum(),
    "Matches within 1e-6": nimtaavg_error.le(1e-6).sum(),
    "Matches within 1e-4": nimtaavg_error.le(1e-4).sum(),
    "Mean absolute error": nimtaavg_error.mean(),
    "Median absolute error": nimtaavg_error.median(),
    "99th percentile error": nimtaavg_error.quantile(0.99),
    "Maximum absolute error": nimtaavg_error.max()
})

display(nimtaavg_validation)

del history, groups

## Baseline eligibility and missing values

Require finite values for all seven non-SIGMA CHS predictors, including `nimtaavg` and `exretavg`; do not invent missing accounting ratios. Allow missing raw SIGMA and later impute it using the winsorized estimation-sample mean, rather than the paper's cross-sectional mean. In the original training run this retained 432 additional positive outcomes relative to complete-case deletion (2,207 versus 1,775). The restriction reduces early accounting coverage substantially; use the same eligible rows for baseline and expanded model comparisons, and treat broader imputation as a possible robustness exercise.

In [ ]:
CHS_REQUIRED = [
    feature for feature in CHS_FEATURES
    if feature != "sigma"
]

finite_chs = (
    df[CHS_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
)

df["baseline_eligible"] = (
    finite_chs[CHS_REQUIRED]
    .notna()
    .all(axis=1)
)

print(
    "Baseline-eligible forecasting observations:",
    df["baseline_eligible"].sum()
)

print(
    "Baseline-eligible labeled observations:",
    (
        df["baseline_eligible"]
        & df["labeled"]
    ).sum()
)

In [ ]:
sample_policies = {
    "Unrestricted": pd.Series(True, index=df.index),

    "Complete case":
        finite_chs.notna().all(axis=1),

    "Selective SIGMA imputation":
        df["baseline_eligible"]
}

period_masks = {
    "Training": train_mask,
    "Validation": validation_mask
}

records = []

for period, period_mask in period_masks.items():

    original = df.loc[period_mask]

    for policy, policy_mask in sample_policies.items():

        sample = df.loc[period_mask & policy_mask]

        records.append({
            "period": period,
            "policy": policy,
            "observations": len(sample),
            "unique_securities": sample["permno"].nunique(),
            "failures": sample["target"].sum(),
            "failure_rate_pct": sample["target"].mean() * 100,

            "observations_retained_pct":
                len(sample) / len(original) * 100,

            "failures_retained_pct":
                sample["target"].sum()
                / original["target"].sum() * 100
        })

policy_comparison = pd.DataFrame(records)

display(policy_comparison)

## Expanded information set

Start with an explicit economically motivated 39-characteristic candidate list rather than all numeric columns. Profitability and liquidity measure internal funding and cash buffers; leverage measures obligations; growth and investment capture deterioration or excessive expansion; size and age proxy for financing access and maturity. Valuation and momentum capture market expectations, volatility captures uncertainty, issuance measures external financing or recapitalization, asset tangibility captures collateral, and trading liquidity reflects market frictions. These are economic motivations, not causal claims or verified vendor definitions. Exclude metadata, future returns and composite distress scores from this initial library.

In [ ]:
FEATURE_GROUPS = {
    "profitability": [
        "gp_at",
        "op_at",
        "niq_at",
        "ebit_sale",
        "ocf_at"
    ],

    "leverage": [
        "at_be",
        "debt_me",
        "netdebt_me"
    ],

    "liquidity": [
        "cash_at",
        "aliq_at",
        "aliq_mat"
    ],

    "growth": [
        "at_gr1",
        "sale_gr1",
        "saleq_gr1",
        "niq_at_chg1"
    ],

    "investment": [
        "capx_gr1",
        "inv_gr1",
        "noa_gr1a"
    ],

    "size": [
        "me_company"
    ],

    "valuation": [
        "be_me",
        "ni_me",
        "ocf_me",
        "ebitda_mev"
    ],

    "momentum": [
        "ret_1_0",
        "ret_6_1",
        "ret_12_1"
    ],

    "volatility": [
        "rvol_21d",
        "ivol_capm_252d",
        "earnings_variability"
    ],

    "issuance": [
        "chcsho_12m",
        "eqnetis_at",
        "dbnetis_at"
    ],

    "intangible_capital": [
        "rd_sale",
        "rd5_at",
        "tangibility"
    ],

    "trading_liquidity": [
        "ami_126d",
        "zero_trades_126d",
        "turnover_126d"
    ],

    "firm_maturity": [
        "age"
    ]
}

GLOBAL_FEATURES = [
    feature
    for features in FEATURE_GROUPS.values()
    for feature in features
]

ALL_FEATURES = CHS_FEATURES + GLOBAL_FEATURES

available_columns = pq.ParquetFile(
    file_path
).schema_arrow.names

missing_columns = [
    col for col in ALL_FEATURES
    if col not in available_columns
]

assert not missing_columns, missing_columns

print("CHS predictors:", len(CHS_FEATURES))
print("Global characteristics:", len(GLOBAL_FEATURES))
print("Total predictors:", len(ALL_FEATURES))

In [ ]:
import gc

del train, train_features, finite_chs

gc.collect()

In [ ]:
global_features = pd.read_parquet(
    file_path,
    columns=KEYS + GLOBAL_FEATURES
)

global_features["month_end"] = pd.to_datetime(
    global_features["month_end"]
)

assert not global_features.duplicated(KEYS).any()

original_rows = len(df)

df = df.merge(
    global_features,
    on=KEYS,
    how="left",
    validate="one_to_one",
    sort=False,
    indicator=True
)

assert len(df) == original_rows
assert df["_merge"].eq("both").all()

df = df.drop(columns="_merge")

del global_features
gc.collect()

# Reconstruct masks following merge
train_mask = (
    df["labeled"]
    & df["forecast_month"].le(TRAIN_END)
)

validation_mask = (
    df["labeled"]
    & df["forecast_month"].gt(TRAIN_END)
    & df["forecast_month"].le(VALIDATION_END)
)

development_mask = train_mask | validation_mask

print("Dataset shape:", df.shape)

print(
    "Memory usage (GB):",
    round(df.memory_usage(deep=True).sum() / 1e9, 3)
)

In [ ]:
feature_to_group = {
    feature: group
    for group, features in FEATURE_GROUPS.items()
    for feature in features
}

baseline_train_mask = (
    train_mask
    & df["baseline_eligible"]
)

baseline_validation_mask = (
    validation_mask
    & df["baseline_eligible"]
)

coverage_records = []

for feature in GLOBAL_FEATURES:

    training_values = df.loc[
        baseline_train_mask, feature
    ].replace([np.inf, -np.inf], np.nan)

    validation_values = df.loc[
        baseline_validation_mask, feature
    ].replace([np.inf, -np.inf], np.nan)

    coverage_records.append({
        "feature": feature,
        "economic_group": feature_to_group[feature],

        "training_nonmissing_pct":
            training_values.notna().mean() * 100,

        "validation_nonmissing_pct":
            validation_values.notna().mean() * 100,

        "training_mean": training_values.mean(),
        "training_std": training_values.std(),
        "training_median": training_values.median(),

        "training_unique_values":
            training_values.nunique()
    })

global_coverage = pd.DataFrame(
    coverage_records
).sort_values(
    "training_nonmissing_pct",
    ascending=False
)

display(global_coverage)

In [ ]:
coverage_plot = global_coverage.sort_values(
    "training_nonmissing_pct"
)

ax = coverage_plot.set_index("feature")[
    [
        "training_nonmissing_pct",
        "validation_nonmissing_pct"
    ]
].plot.barh(
    figsize=(11, 14),
    width=0.8
)

ax.set_xlabel("Nonmissing observations (%)")
ax.set_ylabel("Characteristic")

ax.set_title(
    "Global Characteristics Coverage: "
    "Training vs. Validation"
)

ax.legend(["Training", "Validation"])

plt.tight_layout()
plt.savefig(figures_dir / "global_feature_coverage.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(ax.figure)

In [ ]:
development_sample = df.loc[
    development_mask
    & df["baseline_eligible"],
    ["forecast_month"] + GLOBAL_FEATURES
]

development_periods = pd.cut(
    development_sample["forecast_month"].dt.year,
    bins=[1962, 1972, 1982, 1992, 1995, 2003],
    labels=[
        "1963-1972",
        "1973-1982",
        "1983-1992",
        "1993-1995",
        "1996-2003"
    ]
)

period_coverage = {}

for feature in GLOBAL_FEATURES:

    observed = (
        development_sample[feature]
        .replace([np.inf, -np.inf], np.nan)
        .notna()
    )

    period_coverage[feature] = (
        observed.groupby(
            development_periods,
            observed=True
        ).mean() * 100
    )

period_coverage = pd.DataFrame(
    period_coverage
).T

display(period_coverage.round(2))

del development_sample
gc.collect()

Inspect rank correlations using a fixed seed (42) and a training-only diagnostic subsample. Do not randomly split the modeling panel. Retain correlated features because they may contain incremental information at different horizons; regularization is a possible choice for teammates' expanded models, not an implemented model here.

In [ ]:
correlation_sample = df.loc[
    baseline_train_mask,
    ALL_FEATURES
].sample(
    n=min(200_000, baseline_train_mask.sum()),
    random_state=42
)

correlation_sample = correlation_sample.replace(
    [np.inf, -np.inf],
    np.nan
)

correlation_matrix = correlation_sample.corr(
    method="spearman",
    min_periods=1000
)

chs_correlations = correlation_matrix.loc[
    GLOBAL_FEATURES,
    CHS_FEATURES
]

redundancy_summary = pd.DataFrame({
    "global_feature": GLOBAL_FEATURES,

    "closest_chs_feature":
        chs_correlations.abs().idxmax(axis=1).values,

    "max_absolute_correlation":
        chs_correlations.abs().max(axis=1).values
}).sort_values(
    "max_absolute_correlation",
    ascending=False
)

display(redundancy_summary)

## Verify NIMTAAVG convention

The initial quarterly-block diagnostic did not reproduce the supplied NIMTAAVG. Compare plausible weighting conventions on identical complete histories before deciding whether to retain the supplied feature.

In [ ]:
history = pd.read_parquet(
    file_path,
    columns=[
        "permno",
        "month_end",
        "nimta",
        "nimtaavg"
    ]
)

history["month_end"] = pd.to_datetime(
    history["month_end"]
)

history = (
    history.loc[history["month_end"] < TRAIN_END]
    .sort_values(["permno", "month_end"])
    .reset_index(drop=True)
)

history["month_index"] = (
    history["month_end"].dt.year * 12
    + history["month_end"].dt.month
)

groups = history.groupby("permno", sort=False)

phi = 2 ** (-1 / 3)

monthly_weights = phi ** np.arange(12)
monthly_weights /= monthly_weights.sum()

quarter_weights = phi ** (3 * np.arange(4))
quarter_weights /= quarter_weights.sum()

methods = {
    "Quarterly lags 0, 3, 6, 9": dict(
        zip([0, 3, 6, 9], quarter_weights)
    ),

    "Quarterly lags 2, 5, 8, 11": dict(
        zip([2, 5, 8, 11], quarter_weights)
    ),

    "Monthly geometric weights": dict(
        zip(range(12), monthly_weights)
    ),

    "Quarterly block averages": dict(
        zip(range(12), np.repeat(quarter_weights / 3, 3))
    )
}

reconstructed = {
    name: np.zeros(len(history))
    for name in methods
}

valid = history["nimtaavg"].notna().to_numpy()

for lag in range(12):

    value = groups["nimta"].shift(lag)
    month = groups["month_index"].shift(lag)

    consecutive = (
        history["month_index"] - month
    ).eq(lag)

    valid &= (
        value.notna() & consecutive
    ).to_numpy()

    values = value.fillna(0).to_numpy()

    for name, method_weights in methods.items():
        if lag in method_weights:
            reconstructed[name] += (
                method_weights[lag] * values
            )

records = []

for name, values in reconstructed.items():

    error = np.abs(
        history.loc[valid, "nimtaavg"].to_numpy()
        - values[valid]
    )

    records.append({
        "method": name,
        "comparable_observations": len(error),
        "match_pct_1e6": np.mean(error <= 1e-6) * 100,
        "match_pct_1e4": np.mean(error <= 1e-4) * 100,
        "mean_absolute_error": error.mean(),
        "median_absolute_error": np.median(error),
        "max_absolute_error": error.max()
    })

nimta_methods = pd.DataFrame(records).sort_values(
    "mean_absolute_error"
)

display(nimta_methods)

del history, groups, reconstructed
gc.collect()

In the original run, monthly geometric weighting reproduced supplied NIMTAAVG within $10^{-6}$ for all 869,404 comparable rows; EXRETAVG also matched monthly geometric weighting for complete histories. Retain the supplied features and document the monthly-versus-quarterly NIMTAAVG deviation from CHS. The original historical missing-lag replacement remains unverified.

## Coverage screen

Keep additional features with at least 60% finite coverage on the initial baseline-eligible training sample. This outcome-independent threshold limits excessive imputation while preserving a broad economic library. It excludes `rd_sale` and `rd5_at` in the original run, leaving 37 additional features (45 total). Retain `tangibility` as an asset-structure proxy. Freeze this feature list for the main comparison; no test performance enters the screen.

In [ ]:
MIN_COVERAGE = 60.0

coverage_lookup = global_coverage.set_index(
    "feature"
)["training_nonmissing_pct"]

SELECTED_GLOBAL_FEATURES = [
    feature
    for feature in GLOBAL_FEATURES
    if coverage_lookup[feature] >= MIN_COVERAGE
]

EXCLUDED_GLOBAL_FEATURES = [
    feature
    for feature in GLOBAL_FEATURES
    if feature not in SELECTED_GLOBAL_FEATURES
]

FINAL_FEATURES = (
    CHS_FEATURES + SELECTED_GLOBAL_FEATURES
)

FINAL_FEATURE_GROUPS = {
    group: [
        feature for feature in features
        if feature in SELECTED_GLOBAL_FEATURES
    ]
    for group, features in FEATURE_GROUPS.items()
}

print("CHS features:", len(CHS_FEATURES))

print(
    "Selected Global Characteristics:",
    len(SELECTED_GLOBAL_FEATURES)
)

print(
    "Total predictors:",
    len(FINAL_FEATURES)
)

print(
    "Excluded features:",
    EXCLUDED_GLOBAL_FEATURES
)

display(
    global_coverage.loc[
        global_coverage["feature"].isin(
            EXCLUDED_GLOBAL_FEATURES
        )
    ]
)

In [ ]:
feature_manifest = pd.DataFrame([
    {
        "feature": feature,
        "source": "CHS",
        "economic_group": "baseline",
        "selected": True,
        "missing_value_policy": (
            "training_mean"
            if feature == "sigma"
            else "required"
        )
    }
    for feature in CHS_FEATURES
] + [
    {
        "feature": feature,
        "source": "Global Characteristics",
        "economic_group": feature_to_group[feature],
        "selected": feature in SELECTED_GLOBAL_FEATURES,
        "missing_value_policy": (
            "training_median"
            if feature in SELECTED_GLOBAL_FEATURES
            else "excluded"
        )
    }
    for feature in GLOBAL_FEATURES
])

feature_manifest["lower_winsorization_quantile"] = 0.05
feature_manifest["upper_winsorization_quantile"] = 0.95

display(feature_manifest)

A separate Python file has been created for the GitHub repository with reusable winsorization and missing-value treatment.

The module contains two principal functions:

*fit_preprocessor()*: Estimate winsorization bounds and imputation parameters using the supplied training sample

*transform_features()*: Apply previously estimated parameters without learning anything from new observations

## Preprocessing choices

Treat infinities as missing and clip all selected predictors at the pooled 5th and 95th percentiles fitted on the eligible labeled estimation sample only. This follows the CHS outlier-control approach but avoids full-sample thresholds; PRICE retains the supplied $15 cap before our percentile clipping. After clipping, use the training mean for missing SIGMA and training medians for missing expanded characteristics (robust to skew). Required baseline features are not imputed. Do not add missingness indicators or scale the shared predictors; model-specific scaling can be fitted within training folds. `fit_preprocessor` estimates parameters, while `transform_features` applies them without learning from validation or test data.

In [ ]:
import sys

src_dir = project_root / "src"

assert (src_dir / "preprocessing.py").exists()

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from preprocessing import (
    fit_preprocessor,
    transform_features
)

print("Preprocessing module imported successfully.")

In [ ]:
baseline_train_mask = (
    train_mask
    & df["baseline_eligible"]
)

initial_params = fit_preprocessor(
    train=df.loc[
        baseline_train_mask,
        FINAL_FEATURES
    ],
    chs_features=CHS_FEATURES,
    global_features=SELECTED_GLOBAL_FEATURES,
    lower=0.05,
    upper=0.95
)

display(initial_params)

assert len(initial_params) == 45

assert initial_params.index.tolist() == FINAL_FEATURES

assert initial_params["training_rows"].eq(
    baseline_train_mask.sum()
).all()

print("Initial preprocessing parameters validated.")

In [ ]:
baseline_validation_mask = (
    validation_mask
    & df["baseline_eligible"]
)

train_preview = df.loc[
    baseline_train_mask,
    FINAL_FEATURES
].head(1000)

validation_preview = df.loc[
    baseline_validation_mask,
    FINAL_FEATURES
].head(1000)

X_train_preview = transform_features(
    train_preview,
    initial_params
)

X_validation_preview = transform_features(
    validation_preview,
    initial_params
)

assert X_train_preview.shape == (1000, 45)
assert X_validation_preview.shape == (1000, 45)

assert not X_train_preview.isna().any().any()
assert not X_validation_preview.isna().any().any()

assert np.isfinite(
    X_train_preview.to_numpy()
).all()

assert np.isfinite(
    X_validation_preview.to_numpy()
).all()

for feature in FINAL_FEATURES:

    lower = initial_params.loc[feature, "lower"]
    upper = initial_params.loc[feature, "upper"]

    assert X_validation_preview[feature].between(
        lower, upper
    ).all()

print("All preprocessing integrity checks passed.")

display(X_train_preview.head())

## Preserve raw predictors

Export the untransformed 45-feature panel with eligibility and target metadata, plus the ordered manifest and initial preprocessing parameters. This allows teammates to refit transformations within rolling windows or test alternative missing-value policies rather than inherit a single permanently transformed panel.

In [ ]:
METADATA_COLS = [
    "permno",
    "permco",
    "gvkey",
    "month_end",
    "feature_month",
    "forecast_month",
    "target",
    "labeled",
    "baseline_eligible"
]

export_cols = METADATA_COLS + FINAL_FEATURES

assert len(export_cols) == len(set(export_cols))

assert not df.duplicated(
    ["permno", "month_end"]
).any()

feature_panel_path = (
    processed_dir / "part_II_features.parquet"
)

df[export_cols].to_parquet(
    feature_panel_path,
    index=False,
    compression="snappy"
)

metadata = pq.ParquetFile(
    feature_panel_path
).metadata

print("Part II feature panel exported.")

print("Rows:", metadata.num_rows)
print("Columns:", metadata.num_columns)

print(
    "File size (GB):",
    round(feature_panel_path.stat().st_size / 1e9, 3)
)

print("Path:", feature_panel_path)

In [ ]:
feature_manifest.to_csv(
    tables_dir / "feature_manifest.csv",
    index=False
)

initial_params.to_csv(
    tables_dir / "preprocessing_params_initial.csv"
)

policy_comparison.to_csv(
    tables_dir / "missing_value_policy_comparison.csv",
    index=False
)

global_coverage.to_csv(
    tables_dir / "global_feature_coverage.csv",
    index=False
)

period_coverage.to_csv(
    tables_dir / "historical_feature_coverage.csv"
)

redundancy_summary.to_csv(
    tables_dir / "feature_redundancy.csv",
    index=False
)

nimta_methods.to_csv(
    tables_dir / "nimtaavg_validation.csv",
    index=False
)

group_means.to_csv(
    tables_dir / "chs_failure_group_means.csv"
)

group_medians.to_csv(
    tables_dir / "chs_failure_group_medians.csv"
)

print("Part II diagnostic tables exported successfully.")

# Export the numeric evidence behind the construction and missingness checks.
for filename, diagnostic in {
    "chs_training_feature_summary": feature_summary,
    "chs_missingness": missingness,
    "chs_unusual_values": unusual_values,
    "price_validation": price_validation,
    "exret_validation": exret_validation,
    "exretavg_validation": exretavg_validation,
    "nimtaavg_quarterly_block_validation": nimtaavg_validation,
    "accounting_ratio_validation": accounting_validation,
    "accounting_availability": accounting_diagnostics,
    "sigma_diagnostics": sigma_diagnostics,
    "sigma_fill_error_summary": fill_error.describe(),
    "exret_weights": weight_table,
}.items():
    if isinstance(diagnostic, pd.Series):
        diagnostic.to_csv(tables_dir / f"{filename}.csv", header=["value"])
    else:
        diagnostic.to_csv(tables_dir / f"{filename}.csv", index=filename != "exret_weights")
print("Additional validation tables and coverage figure saved.")